# Pipeline SDC — exécution interactive

Notebook pour lancer le pipeline `metadata-analysis-llm-for-sdc` (branche `2-modifications-jawad`)
sans repasser par la ligne de commande, et pour évaluer la qualité d'un output contre un CSV de référence.

**Prérequis** : variable d'environnement `OPENAI_API_KEY` ou `CLE_API_OPENWEBUI` définie (clé LLM SSP Cloud).
Si vos fichiers d'entrée/référence sont sur MinIO (`s3://...`), il faut aussi `AWS_ACCESS_KEY_ID`,
`AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`.


In [11]:
import sys
from pathlib import Path

# Détecte la racine du repo (fonctionne que le notebook soit à la racine ou dans un sous-dossier)
for _p in [Path.cwd(), Path.cwd().parent]:
    if (_p / "src").is_dir():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Impossible de localiser 'src/'. Placez ce notebook a la racine du repo (ou un niveau en dessous).")

sys.path.insert(0, str(REPO_ROOT))

from src.data import read_file
from src.clean import _dataframe_to_rows, clean_sheet
from src.transform_input import wrap, to_markdown
from src.LLM_API_call import chat, is_auto_continued
from src.extract_JSON_array import extract_array
from src.validate_json_output import validate
from src.transform_output import _spanning_pairs, max_spanning, HEADER_BASE

print(f"REPO_ROOT = {REPO_ROOT}")


REPO_ROOT = /home/onyxia/work/metadata-analysis-llm-for-sdc


## Configuration

Renseignez le fichier d'entree, le chemin de sortie, et (optionnel) un CSV de reference
pour l'evaluation de qualite en fin de notebook.


In [24]:
INPUT_PATH  = "s3://jawadmallat/analyse_LLM_metadata/data_tables/sujets_analyse_demande_1.ods"     # .ods / .xlsx / .csv, local ou s3://...
OUTPUT_PATH = "s3://jawadmallat/analyse_LLM_metadata/6juillet/sujets_analyse_demande_1.csv"          # sortie du pipeline
PROMPT_PATH = "src/prompts/prompt_questions.md"

# Optionnel : CSV de reference (sortie attendue, corrigee a la main) pour l'evaluation de qualite.
# Laisser a None pour sauter cette etape.
REFERENCE_CSV = "s3://jawadmallat/analyse_LLM_metadata/data_tables/correction/sujets_analyse_demande_1_correction.csv"   # ex: "path/to/correction.csv"


## I-III. Lecture, nettoyage, mise en Markdown


In [25]:
from IPython.display import Markdown, display

# ---- I. Read ---------------------------------------------------
data = read_file(INPUT_PATH)

# ---- II. Clean ---------------------------------------------------
cleaned_sheets = []
for name, sheet_df in data.items():
    rows = _dataframe_to_rows(sheet_df)
    rows = clean_sheet(rows)
    if any(any(c for c in r) for r in rows):
        cleaned_sheets.append((name, rows))

# ---- III. Transform to markdown ------------------------------------
title = INPUT_PATH.rsplit("/", 1)[-1]
markdown_input = to_markdown(cleaned_sheets, title=title)

print(f"{len(markdown_input)} caracteres -- apercu :")
display(Markdown(markdown_input))


1475 caracteres -- apercu :


# sujets_analyse_demande_1.ods


## Demande_n°1

| Tab | Var_reponse | expl_var | expl_var.1 |
| --- | --- | --- | --- |
| T1 | ca_pizzas | Nuts2 | TREFF |
| T2 | ca_pizzas | Nuts3 | TREFF |
| T3 | ca_pizzas | A10 | Nuts2 |
| T4 | ca_pizzas | A10 | Nuts3 |
| T5 | ca_pizzas | A21 | Nuts2 |
| T6 | ca_pizzas | A21 | Nuts3 |
| T7 | ca_pizzas | A88 | Nuts2 |
| T8 | ca_pizzas | A88 | Nuts3 |
| T9 | ca_batavia | A10 | TREFF |
| T10 | ca_batavia | A10 | CJ |
| T11 | ca_batavia | A21 | TREFF |
| T12 | ca_batavia | A21 | CJ |
| T13 | ca_batavia | A88 | TREFF |
| T14 | ca_batavia | A88 | CJ |
| T15 | ca_mache | A10 | TREFF |
| T16 | ca_mache | A10 | CJ |
| T17 | ca_mache | A21 | TREFF |
| T18 | ca_mache | A21 | CJ |
| T19 | ca_mache | A88 | TREFF |
| T20 | ca_mache | A88 | CJ |
| T21 | ca_salades | A10 | TREFF |
| T22 | ca_salades | A10 | CJ |
| T23 | ca_salades | A21 | TREFF |
| T24 | ca_salades | A21 | CJ |
| T25 | ca_salades | A88 | TREFF |
| T26 | ca_salades | A88 | CJ |
| Pour chaque tableau concernant l’activité |  |  |  |
| On souhaite disposer des deux sous totaux supplémentaires : |  |  |  |
| D_To_H | D+E+F+G+H | Code NAF : A21 |  |
| C14_To_C19 | 14 + 15 +16 +18 +19 | Code NAF : A88 |  |
|  |  |  |  |
| Informations complémentaires : |  |  |  |
| Il n’ y a que deux types de salades : Les Batavias et La Mâche |  |  |  |
|  |  |  |  |
| champ de la population |  |  |  |
| Tous les tableaux portent sur le même champ des entreprises françaises |  |  |  |


## IV. Phase 1 — le modele analyse et pose ses questions

`chat()` envoie le prompt systeme + les metadonnees. Le modele repond soit directement
en JSON (aucune question), soit par une liste de questions.


In [26]:
prompt = open(PROMPT_PATH, encoding="utf-8").read()
history = [
    {"role": "system", "content": prompt},
    {"role": "user", "content": wrap(markdown_input)},
]

print("Phase 1 -- envoi au modele...\n")
reply = chat(history)
history.append({"role": "assistant", "content": reply})

auto_continued = is_auto_continued(reply)

if auto_continued:
    print("Le modele n'a pose aucune question -- JSON produit directement.")
    print("Passez directement a la cellule 'Verification' (la cellule 'Reponses' sera ignoree).")
else:
    print("-" * 70)
    print(reply)
    print("-" * 70)
    print("\n-> Remplissez la variable `answers` dans la cellule suivante, puis executez-la.")


Phase 1 -- envoi au modele...

----------------------------------------------------------------------
**Liste des feuilles du fichier :**

1.  **Demande_n°1** : Feuille de demande (contient les tableaux T1 à T26, les notes sur les hiérarchies et le champ).

---

### 1. Champ et population

1. Pour la variable `ca_batavia`, s'agit-il du chiffre d'affaires des Batavias uniquement, ou inclut-elle d'autres types de salades ? La note indique « Il n’y a que deux types de salades : Les Batavias et La Mâche », mais ne précise pas si `ca_batavia` est un sous-ensemble de `ca_salades` ou une variable indépendante.
2. Pour la variable `ca_mache`, s'agit-il du chiffre d'affaires de la Mâche uniquement, ou inclut-elle d'autres types de salades ? La note indique « Il n’y a que deux types de salades : Les Batavias et La Mâche », mais ne précise pas si `ca_mache` est un sous-ensemble de `ca_salades` ou une variable indépendante.

### 2. Indicateurs et hiérarchies

3. La note « Il n’y a que deux types d

## Reponses du producteur

Ne remplir que si des questions ont ete posees ci-dessus (une reponse par question, dans l'ordre).


In [28]:
answers = """
1) Oui, uniquement  
2) Oui, uniquement
3) Oui
4) Non
5) hrc dans ce cas: NA 
6) hrc dans ce cas: NA
7) Oui, hrc_naf 
8) Oui, hrc_nuts
"""

print("Reponses enregistrees -- executez la cellule 'Verification' pour envoyer au modele.")


Reponses enregistrees -- executez la cellule 'Verification' pour envoyer au modele.


## V-VI. Verification et ecriture du CSV

Envoie les reponses si necessaire, valide le JSON contre le schema, puis ecrit directement le CSV.
Une seule cellule a executer une fois que le modele a fini de repondre.


In [29]:
if not auto_continued:
    print("Phase 2 -- envoi des reponses au modele...\n")
    history.append({"role": "user", "content": answers})
    reply = chat(history)

# ---- V. Verify -----------------------------------------------------
records = extract_array(reply)
if records is None:
    raise ValueError("Aucun tableau JSON trouve dans la reponse du modele.")

errors = validate(records)
if errors:
    raise ValueError("Schema validation failed:\n" + "\n".join(errors))

print(f"{len(records)} enregistrement(s) valides contre le schema.")

# ---- VI. Write CSV ---------------------------------------------------
import csv
import pandas as pd

def _open_any(path, mode="r", **kw):
    """open() local ou s3fs selon le prefixe du chemin."""
    if path.startswith("s3://"):
        from src.data import connect_s3
        return connect_s3().open(path, mode, **kw)
    return open(path, mode, **kw)

n_span = max_spanning(records)
cols = list(HEADER_BASE)
for i in range(1, n_span + 1):
    cols += [f"spanning_{i}", f"hrc_spanning_{i}"]

rows = []
for rec in records:
    row = [rec["table_name"], rec["field"], rec["hrc_field"],
           rec["indicator"], rec["hrc_indicator"]]
    pairs = _spanning_pairs(rec)
    for i in range(n_span):
        code_, hrc = pairs[i] if i < len(pairs) else ("", "")
        row += [code_, hrc]
    rows.append(row)

with _open_any(OUTPUT_PATH, "w", newline="", encoding="utf-8-sig") as fh:
    w = csv.writer(fh)
    w.writerow(cols)
    w.writerows(rows)

print(f"Ecrit {OUTPUT_PATH} ({len(rows)} lignes x {len(cols)} colonnes)")

with _open_any(OUTPUT_PATH, "r", encoding="utf-8-sig") as f:
    df = pd.read_csv(f, keep_default_na=False)
display(df)


Phase 2 -- envoi des reponses au modele...

26 enregistrement(s) valides contre le schema.
Ecrit s3://jawadmallat/analyse_LLM_metadata/6juillet/sujets_analyse_demande_1.csv (26 lignes x 9 colonnes)


,table_name,field,hrc_field,indicator,hrc_indicator,spanning_1,hrc_spanning_1,spanning_2,hrc_spanning_2
0,T1,entreprises_francaises,NA,ca_pizzas,NA,Nuts2,hrc_nuts,TREFF,NA
1,T2,entreprises_francaises,NA,ca_pizzas,NA,Nuts3,hrc_nuts,TREFF,NA
2,T3,entreprises_francaises,NA,ca_pizzas,NA,A10,hrc_naf,Nuts2,hrc_nuts
3,T4,entreprises_francaises,NA,ca_pizzas,NA,A10,hrc_naf,Nuts3,hrc_nuts
4,T5,entreprises_francaises,NA,ca_pizzas,NA,A21,hrc_naf,Nuts2,hrc_nuts
5,T6,entreprises_francaises,NA,ca_pizzas,NA,A21,hrc_naf,Nuts3,hrc_nuts
6,T7,entreprises_francaises,NA,ca_pizzas,NA,A88,hrc_naf,Nuts2,hrc_nuts
7,T8,entreprises_francaises,NA,ca_pizzas,NA,A88,hrc_naf,Nuts3,hrc_nuts
8,T9,entreprises_francaises,NA,ca_batavia,hrc_salades,A10,hrc_naf,TREFF,NA
9,T10,entreprises_francaises,NA,ca_batavia,hrc_salades,A10,hrc_naf,CJ,NA


## Evaluation de qualite

Compare la sortie du pipeline avec `REFERENCE_CSV` (sortie attendue, corrigee a la main) et
compte les cellules incorrectes, colonne par colonne. Chaque execution ajoute une ligne dans
`notebook/results.csv` pour suivre si le modele se trompe systematiquement sur une variable
donnee plutot qu'aleatoirement.

Necessite `REFERENCE_CSV` renseigne dans la cellule de configuration ci-dessus.


In [30]:
import re
from datetime import datetime, timezone

if REFERENCE_CSV is None:
    print("REFERENCE_CSV n'est pas renseigne -- etape sautee.")
else:
    def _open_any(path, mode="r", **kw):
        if path.startswith("s3://"):
            from src.data import connect_s3
            return connect_s3().open(path, mode, **kw)
        return open(path, mode, **kw)

    RESULTS_CSV = REPO_ROOT / "notebook" / "results.csv"
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

    with _open_any(REFERENCE_CSV, "r", encoding="utf-8-sig") as f:
        ref = pd.read_csv(f, keep_default_na=False)
    out = df.copy()

    # -- Comparaison du nombre de tableaux --------------------------------
    n_tables_ref = len(ref)
    n_tables_out = len(out)
    n_tables_diff = n_tables_out - n_tables_ref

    # -- Alignement sur table_name (repli positionnel) --------------------
    def _align(ref_df, out_df):
        if "table_name" in ref_df.columns and "table_name" in out_df.columns:
            ref_ids = ref_df["table_name"].tolist()
            out_lookup = set(out_df["table_name"])
            common = [t for t in ref_ids if t in out_lookup]
            if common:
                ra = ref_df.set_index("table_name").reindex(common)
                oa = out_df.set_index("table_name").reindex(common)
                return ra, oa, len(ref_df) - len(common), len(out_df) - len(common)
        n = min(len(ref_df), len(out_df))
        return (ref_df.iloc[:n].reset_index(drop=True),
                out_df.iloc[:n].reset_index(drop=True),
                len(ref_df) - n, len(out_df) - n)

    ref_a, out_a, n_miss_ref, n_miss_out = _align(ref, out)

    # Normalise les cellules vides en "NA" (sentinel du schema)
    ref_a = ref_a.fillna("NA").replace("", "NA")
    out_a = out_a.fillna("NA").replace("", "NA")

    # -- Comptage des cellules incorrectes par colonne ---------------------
    SPAN_RE = re.compile(r"^spanning_\d+$")
    HRC_SPAN_RE = re.compile(r"^hrc_spanning_\d+$")

    col_errors = {"table_name": 0, "field": 0, "hrc_field": 0,
                  "indicator": 0, "hrc_indicator": 0,
                  "spanning": 0, "hrc_spanning": 0}

    for col in ref_a.columns:
        if col not in out_a.columns:
            continue
        n_wrong = int((ref_a[col].astype(str) != out_a[col].astype(str)).sum())
        if col in col_errors:
            col_errors[col] += n_wrong
        elif SPAN_RE.match(col):
            col_errors["spanning"] += n_wrong
        elif HRC_SPAN_RE.match(col):
            col_errors["hrc_spanning"] += n_wrong

    unmatched_err = (n_miss_ref + n_miss_out) * max(len(ref_a.columns), 1)
    total_wrong = sum(col_errors.values()) + unmatched_err

    # -- Resume -------------------------------------------------------------
    diff_label = "=" if n_tables_diff == 0 else f"{n_tables_diff:+d}"
    print(f"  Tableaux attendus  : {n_tables_ref}")
    print(f"  Tableaux produits  : {n_tables_out}  [{diff_label}]")
    if n_tables_diff != 0:
        print(f"  !! LLM a {'ajoute' if n_tables_diff > 0 else 'omis'} {abs(n_tables_diff)} tableau(x)")
    print(f"\n  {'Colonne':<22} {'Cellules incorrectes':>20}")
    print("  " + "-" * 44)
    for k, v in col_errors.items():
        print(f"  {k:<22} {v:>20}")
    if n_miss_ref or n_miss_out:
        print(f"\n  Lignes manquantes : {n_miss_ref}")
        print(f"  Lignes en exces   : {n_miss_out}")
    print(f"\n  -> TOTAL : {total_wrong} cellules incorrectes")

    # -- results.csv ----------------------------------------------------------
    RESULT_COLS = [
        "filename", "timestamp", "total_wrong_cells",
        "n_tables_ref", "n_tables_out", "n_tables_diff",
        "table_name", "field", "hrc_field",
        "indicator", "hrc_indicator",
        "spanning", "hrc_spanning",
        "missing_from_output", "extra_in_output",
    ]
    row = {
        "filename": Path(INPUT_PATH).name,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "total_wrong_cells": total_wrong,
        "n_tables_ref": n_tables_ref,
        "n_tables_out": n_tables_out,
        "n_tables_diff": n_tables_diff,
        **col_errors,
        "missing_from_output": n_miss_ref,
        "extra_in_output": n_miss_out,
    }

    write_header = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=RESULT_COLS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        w.writerow(row)


  Tableaux attendus  : 26
  Tableaux produits  : 26  [=]

  Colonne                Cellules incorrectes
  --------------------------------------------
  table_name                                0
  field                                     0
  hrc_field                                 0
  indicator                                 0
  hrc_indicator                             0
  spanning                                  0
  hrc_spanning                              0

  -> TOTAL : 0 cellules incorrectes
